# 01 – Nettoyage et vérification (Sujet 9 : Groupe Valmont)

Objectif : passer de 940 lignes brutes à 800 lignes sans doublons, puis à 794 lignes pour la modélisation, en documentant chaque décision.

In [47]:
import pandas as pd
import numpy as np

hotels = pd.read_csv('../data/raw/Valmont_hotels.csv')
reserv = pd.read_csv('../data/raw/Valmont_reservations.csv')
print(hotels.shape, reserv.shape)
reserv.head()

(32, 6) (940, 13)


,reservation_id,hotel_id,segment_declare,nb_nuits,nb_personnes,delai_reservation_j,mois_sejour,weekend,canal,prix_nuit_eur,petit_dejeuner,nuitees_cumulees_client,a_annule
0,R700357,H45,groupe,1,11.0,15.0,9,non,OTA,80.33,non,9.0,non
1,R700658,H51,loisir,2,4.0,9.0,8,oui,direct,157.02,oui,4.0,non
2,R700649,H41,affaires,5,1.0,7.0,12,oui,direct,434.92,oui,6.0,non
3,R700732,H69,loisir,4,1.0,83.0,2,non,OTA,92.70,non,1.0,oui
4,R700098,H52,loisir,3,6.0,213.0,3,oui,OTA,85.74,oui,8.0,non


## 1. Inspection
On regarde les types, les manquants et les valeurs extrêmes avant de toucher à quoi que ce soit.

In [38]:
print(reserv.dtypes)
print(reserv.isna().sum())
reserv.describe().round(2)



reservation_id                 str
hotel_id                       str
segment_declare                str
nb_nuits                     int64
nb_personnes               float64
delai_reservation_j        float64
mois_sejour                  int64
weekend                        str
canal                          str
prix_nuit_eur              float64
petit_dejeuner                 str
nuitees_cumulees_client    float64
a_annule                       str
dtype: object
reservation_id              0
hotel_id                    0
segment_declare             0
nb_nuits                    0
nb_personnes               24
delai_reservation_j         0
mois_sejour                 0
weekend                     0
canal                       0
prix_nuit_eur               0
petit_dejeuner             24
nuitees_cumulees_client     0
a_annule                    0
dtype: int64


,nb_nuits,nb_personnes,delai_reservation_j,mois_sejour,prix_nuit_eur,nuitees_cumulees_client
count,940.00,916.00,940.00,940.00,940.00,940.00
mean,4.10,3.41,42.31,6.48,193.58,11.23
std,20.48,4.05,40.55,3.47,105.17,8.27
min,1.00,1.00,0.00,1.00,0.00,1.00
25%,2.00,1.00,10.00,3.00,115.48,5.00
50%,3.00,1.00,28.00,6.00,152.36,9.00
75%,4.00,3.00,64.25,10.00,280.97,15.00
max,365.00,18.00,235.00,12.00,481.09,54.00


## 2. Doublons (940 → 800)
On compte d'abord les doublons exacts, puis on vérifie que les identifiants répétés correspondent bien à des lignes strictement identiques (sinon ce serait un autre problème qu'un simple doublon).

In [ ]:
display(reserv[reserv.duplicated(keep=False)].sort_values('reservation_id'))

,reservation_id,hotel_id,segment_declare,nb_nuits,nb_personnes,delai_reservation_j,mois_sejour,weekend,canal,prix_nuit_eur,petit_dejeuner,nuitees_cumulees_client,a_annule
372,R700003,H40,loisir,4,1.0,77.0,4,non,OTA,124.79,non,18.0,non
215,R700003,H40,loisir,4,1.0,77.0,4,non,OTA,124.79,non,18.0,non
718,R700008,H42,affaires,2,1.0,24.0,5,oui,direct,379.36,non,4.0,oui
346,R700008,H42,affaires,2,1.0,24.0,5,oui,direct,379.36,non,4.0,oui
629,R700016,H52,loisir,2,3.0,25.0,3,oui,OTA,133.69,non,12.0,non
...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,R700779,H47,loisir,4,2.0,83.0,11,non,direct,94.05,non,12.0,non
48,R700784,H54,loisir,4,NaN,74.0,4,non,OTA,134.90,non,1.0,oui
844,R700784,H54,loisir,4,NaN,74.0,4,non,OTA,134.90,non,1.0,oui
911,R700787,H70,affaires,2,1.0,3.0,5,non,direct,134.83,oui,40.0,non


In [ ]:
n_exacts = reserv.duplicated().sum()
n_id = reserv['reservation_id'].duplicated().sum()
print('doublons exacts :', n_exacts, '| identifiants répétés :', n_id)
# si les deux nombres sont égaux, chaque id répété est un doublon exact

doublons exacts : 140 | identifiants répétés : 140
lignes après dédoublonnage : 800


In [49]:
assert n_exacts == n_id
df = reserv.drop_duplicates().reset_index(drop=True)
print('lignes après dédoublonnage :', len(df))

lignes après dédoublonnage : 800


## 3. Cohérence avec la table des hôtels
Chaque réservation doit pointer vers un hôtel connu (clé de jointure pour Q3, Q4, Q6, Q9).

In [42]:
print('hotel_id inconnus :', set(df.hotel_id) - set(hotels.hotel_id))
print('hôtels sans réservation :', len(set(hotels.hotel_id) - set(df.hotel_id)))
print(hotels.categorie.value_counts())

hotel_id inconnus : set()
hôtels sans réservation : 0
categorie
3 étoiles    13
4 étoiles    11
5 étoiles     8
Name: count, dtype: int64


## 4. Valeurs manquantes
Seules `nb_personnes` (20) et `petit_dejeuner` (20) sont concernées.

**Décisions :**
- `nb_personnes` : médiane **par segment déclaré**, car la taille d'un séjour dépend très fortement du segment (groupe ≈ 11, loisir ≈ 2, affaires ≈ 1). Une médiane globale (1) sous-estimerait les groupes.
- `petit_dejeuner` : modalité `inconnu` plutôt qu'une valeur inventée (variable binaire sans règle d'imputation fiable).
- On ajoute un indicateur `nb_personnes_impute` pour pouvoir tester la sensibilité plus tard.

In [43]:
print(df.groupby('segment_declare')['nb_personnes'].median())
df['nb_personnes_impute'] = df['nb_personnes'].isna()
df['nb_personnes'] = df['nb_personnes'].fillna(
    df.groupby('segment_declare')['nb_personnes'].transform('median'))
df['petit_dejeuner'] = df['petit_dejeuner'].fillna('inconnu')
print(df[['nb_personnes', 'petit_dejeuner']].isna().sum())

segment_declare
affaires     1.0
groupe      11.0
loisir       2.0
Name: nb_personnes, dtype: float64
nb_personnes      0
petit_dejeuner    0
dtype: int64


## 5. Valeurs aberrantes à signaler : 365 nuits et prix à 0 €
On **ne supprime pas en silence** : on les isole dans un fichier à part (réutilisé en Q9) et on les exclut seulement de la base de modélisation.

In [44]:
masque = (df.nb_nuits == 365) | (df.prix_nuit_eur == 0)
print('séjours de 365 nuits :', (df.nb_nuits == 365).sum())
print('prix à 0 EUR :', (df.prix_nuit_eur == 0).sum())
print('lignes concernées (union) :', masque.sum())
anomalies = df[masque].copy()
anomalies['motif'] = np.where(anomalies.nb_nuits == 365, '365 nuits', 'prix 0 EUR')
anomalies

séjours de 365 nuits : 3
prix à 0 EUR : 3
lignes concernées (union) : 6


,reservation_id,hotel_id,segment_declare,nb_nuits,nb_personnes,delai_reservation_j,mois_sejour,weekend,canal,prix_nuit_eur,petit_dejeuner,nuitees_cumulees_client,a_annule,nb_personnes_impute,motif
99,R700355,H45,affaires,3,1.0,20.0,3,oui,OTA,0.00,non,5.0,non,False,prix 0 EUR
146,R700232,H63,loisir,365,2.0,59.0,3,non,entreprise,346.25,oui,10.0,oui,False,365 nuits
537,R700103,H41,affaires,1,1.0,5.0,1,oui,direct,0.00,oui,4.0,non,False,prix 0 EUR
572,R700450,H64,loisir,365,3.0,40.0,2,non,agence,140.57,oui,15.0,non,False,365 nuits
678,R700247,H63,loisir,3,6.0,49.0,5,non,direct,0.00,non,1.0,oui,False,prix 0 EUR
785,R700391,H60,affaires,365,1.0,5.0,6,oui,OTA,131.91,non,2.0,non,False,365 nuits


## 6. Colonne `nuitees_cumulees_client`
Un cumul client devrait être **supérieur ou égal** au nombre de nuits de la réservation en cours. On mesure les cas incohérents.

**Décision :** on garde la colonne dans le fichier mais on ne l'utilise pas sans vérification. Pour Q5, on compare le modèle avec et sans (risque de fuite d'information, la valeur pouvant être calculée après coup).

In [45]:
df['cumul_incoherent'] = df['nuitees_cumulees_client'] < df['nb_nuits']
print('lignes incohérentes :', df['cumul_incoherent'].sum())
print(df.loc[df.cumul_incoherent, ['nb_nuits', 'nuitees_cumulees_client', 'a_annule']].head())
print(df.groupby('cumul_incoherent')['a_annule'].apply(lambda s: (s == 'oui').mean()))

lignes incohérentes : 78
    nb_nuits  nuitees_cumulees_client a_annule
3          4                      1.0      oui
47         4                      3.0      oui
48         4                      1.0      oui
55         4                      2.0      non
76         7                      2.0      non
cumul_incoherent
False    0.253463
True     0.333333
Name: a_annule, dtype: float64


## 7. Fichiers de sortie et récapitulatif

In [46]:
modelisation = df[~masque].copy()
df.to_csv('../data/clean/reservations_propres_800.csv', index=False)
modelisation.to_csv('../data/clean/reservations_modelisation_794.csv', index=False)
anomalies.to_csv('../data/clean/anomalies_a_part.csv', index=False)

recap = pd.DataFrame({'étape': ['Lignes brutes', 'Après suppression des doublons', 'Base de modélisation'],
                      'lignes': [len(reserv), len(df), len(modelisation)]})
print(recap)
print('Taux global d\'annulation (800) :', round((df.a_annule == 'oui').mean(), 3))


                            étape  lignes
0                   Lignes brutes     940
1  Après suppression des doublons     800
2            Base de modélisation     794
Taux global d'annulation (800) : 0.261
